In [ ]:
#gradio: for creating web interfaces for AI models (speech, audio, text, video, pictures, etc.).
#sign up or in on huggingface.co and create a new space to host your gradio app for free. You can also run it locally on your machine.
#clone the repo locally, develop your gradio app, and push it to your huggingface space to share it with the world.
#locally: uv init, uv add gradio
#create a new python file for your gradio app, e.g., app.py, and write your gradio code there.
#example gradio code:
#import gradio as gr
#def hello(name):
#    return f"Hello {name}!" 
#gr.Interface(fn=hello, inputs="text", outputs="text").launch()
#run your app locally with: uv run gradio app.py

In [ ]:
#for deployment: create a requirements.txt: uv pip compile pyproject.toml -o requirements.txt
#git add app.py requirements.txt
#git commit -m "Initial commit of my gradio app"
#git push
#share the link in slack

In [ ]:
#make an image classifier app
#uv add torch torchvision pillow
#example code:
import gradio as gr
import torch
from PIL import Image
from torchvision import models

model=None
preprocess=None
labels = None

def load_model():
    global model, preprocess, labels
    if model is not None:
        return
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    model.eval()
    preprocess = weights.transforms()

    labels = weights.meta["categories"]
@torch.inference_mode()
def predict(img: Image.Image):
    load_model()
    if model is None:
        return {"(no image)": 1.0}
    #load_model()
    x = preprocess(img).unsqueeze(0)
    logits = model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0)
    topk = torch.topk(probs, k=5)
    out={}
    for idx, score in zip(topk.indices.tolist(), topk.values.tolist()):
        out[labels[idx]] = float(score)
    return out 
    img = preprocess(img).unsqueeze(0)
    pred = model(img).argmax().item()
    return labels[pred]

def show_image(img):
    return img


def hello(name):
    return f"Hello {name}!"


#webcam input:
demo=gr.Interface(fn=predict, inputs=gr.Image(sources=["webcam"],type="pil",label="Upload a very interesting image"), outputs=gr.Label(num_top_classes=5, label="Top predictions"),examples=[["examples/Elefant.jpg"], ["examples/Gitarre.jpg"]],title="ImageNet Demo (ResNet18)", description="Upload an image to see and watch all the best and great top 5 predicted classes.")
#upload images input:
#demo = gr.Interface(fn=predict, inputs=gr.Image(type="pil",label="Upload a very interesting image"), outputs=gr.Label(num_top_classes=5, label="Top predictions"),examples=[["examples/Elefant.jpg"], ["examples/Gitarre.jpg"]],title="ImageNet Demo (ResNet18)", description="Upload an image to see and watch all the best and great top 5 predicted classes.")
demo.launch()